<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/04.%20Dataset%20Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Dataset Summary & Paper Auditor**
> MARC - Multidisciplinary AI Research Centre | UOP

This notebook serves as the comprehensive **data profiling, audit, and publication descriptor tool** for the **Wearable Biomechanics** project. It interfaces with Google Drive storage using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** library.

### 🎯 Purpose & Publication Scope
Designed for generating reproducible, publication-ready dataset companion materials for high-impact journals (*Nature Scientific Data*, *Elsevier Data in Brief*, *Sports Biomechanics*):
1. 👤 **Participant Cohort Profiling:** Anthropometrics, demographics, sporting experience, BMI, and missingness metrics.
2. 💾 **Storage Footprint & Modality Breakdown:** Storage footprint, directory breakdowns, and file type distribution (.mp4, .csv, .zip, .parquet).
3. 🧪 **Per-Experiment Telemetry & Hardware Audit:**
   - Raw IMU signal verification across all naming conventions (`Device_1_IMU1_AX`, `OrbiX-Node-1_IMU1_AX`, `stamp` / `Timestamp`).
   - Complete 10-device network and multi-IMU node verification (Devices 1..4 with 2 IMUs; Devices 5..10 with 3 IMUs; total 26 nodes with 6-DoF acceleration + angular velocity axes).
   - Real-time timestamp delta sampling rate estimation, NaN percentage, and dead/flatline channel detection.
   - Related files completeness (raw device zip, synchronized front & side video coverage).
   - Processed data tags and researcher note anomaly extraction (sensor detachments, missing hardware).
4. 📄 **Scientific Deliverables:**
   - `dataset_summary_for_paper.md` (Master Markdown report with formatted tables)
   - `experiments_detailed_audit.csv` (Full per-trial audit spreadsheet with IMU breakdown)
   - `subjects_summary.csv` (Cohort demographics and session totals)
   - `sensor_completeness_matrix.csv` (10-device overview + granular 26-node IMU availability grid)
   - `dataset_stats.json` (Machine-readable metadata descriptor)
   - `dataset_paper_bundle.zip` (All-in-one download package)

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)
* **💻 Project Code:** [Wearable-Biomechanics Repository](https://github.com/janithcyapa/Wearable-Biomechanics/)
* **📂 Data Storage:** [Project Google Drive Folder](https://drive.google.com/drive/folders/1QERF02T4pvubAEjnv6f1VrCOWNYxHv54?usp=drive_link)

## **📚 1. Initial Setup & Google Drive Mount**
Mount your Google Drive and connect to the shared `Wearable-Biomechanics` dataset folder. Ensure you have added the shared folder shortcut to your Drive under `My Drive/Datasets/Wearable-Biomechanics`.

In [2]:
# @title 🛠️ Environment & OpenDatasetStore Setup { display-mode: "form" }
# @markdown Mount Google Drive, install `OpenDatasetStore`, and initialize the dataset connection.
import os
import sys
import re
import json
import time
import zipfile
import shutil
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

# Check if running in Google Colab
try:
    from google.colab import drive, files, data_table
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False

# Install OpenDatasetStore
!pip install --quiet git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

# @markdown ### 📁 Storage Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
output_directory = "./dataset_paper_summary" # @param {type:"string"}

os.makedirs(output_directory, exist_ok=True)

# Initialize OpenDatasetStore instance
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

print("✅ OpenDatasetStore successfully connected to:", store.base_dir)

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.0 MB/s eta 0:00:0000:0100:01
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)
✅ OpenDatasetStore successfully connected to: /content/drive/MyDrive/Datasets/Wearable-Biomechanics


## **💾 2. Storage Volume & Modality Inventory**
Inspect the physical storage utilization across raw data, synchronized video archives, and processed parquets.

In [3]:
# @title 📊 Inspect Storage Footprint { display-mode: "form" }
# @markdown View directory sizes, file counts, and file type distributions.

folder_depth = 2 # @param {type:"integer"}

storage_stats = store.storage_summary(by_folder=True, depth=folder_depth, by_file_type=True)

💾 Storage Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
Total Storage  : 29.35 GB (31,514,403,143 bytes)
Total Files    : 293
-----------------------------------------------------------------
📁 Folder Breakdown (Depth: 2)
-----------------------------------------------------------------
Folder                               Size    Files    % Store
-----------------------------------------------------------------
related_files/experiments        28.90 GB      166      98.5%
raw_data/experiments            352.61 MB       38       1.2%
processed_data/experiments       73.88 MB       78       0.2%
raw_data/device                  35.21 MB        1       0.1%
summary_reports                  51.52 KB        6       0.0%
index                            51.16 KB        3       0.0%
. (root)                          6.15 KB        1       0.0%
-----------------------------------------------------------------
📄 File Type Breakdown
------

## **⚙️ 3. Verification Engine & Anatomical Mapping Specification**
This section defines the anatomical mapping for all 10 body devices, the nominal multi-IMU network (Devices 1..4 with 2 IMUs; Devices 5..10 with 3 IMUs), and the regex parser supporting multiple column conventions (`Device_1_IMU1_AX`, `OrbiX-Node-1_IMU1_AX`, `stamp` / `Timestamp`).

In [4]:
# @title 🧩 Core Auditor Class & Multi-IMU Sensor Network Specifications
# @markdown Configures the 10-device network, IMU counts (1, 2, or 3 IMUs per device), and the robust column parser.

TARGET_ANATOMICAL_MAP = {
    1: {"code": "RKP", "name": "Right Knee Proximal (Thigh)", "role": "Core", "expected_imus": [1, 2]},
    2: {"code": "REP", "name": "Right Elbow Proximal (Upper Arm)", "role": "Core", "expected_imus": [1, 2]},
    3: {"code": "LKP", "name": "Left Knee Proximal (Thigh)", "role": "Core", "expected_imus": [1, 2]},
    4: {"code": "LEP", "name": "Left Elbow Proximal (Upper Arm)", "role": "Core", "expected_imus": [1, 2]},
    5: {"code": "PEL", "name": "Lower Back / Pelvis", "role": "Axial/Extended", "expected_imus": [1, 2, 3]},
    6: {"code": "RED", "name": "Right Elbow Distal (Forearm/Wrist)", "role": "Core", "expected_imus": [1, 2, 3]},
    7: {"code": "LED", "name": "Left Elbow Distal (Forearm/Wrist)", "role": "Core", "expected_imus": [1, 2, 3]},
    8: {"code": "LKD", "name": "Left Knee Distal (Shank)", "role": "Core", "expected_imus": [1, 2, 3]},
    9: {"code": "SPN", "name": "Spine / Upper Back", "role": "Axial/Extended", "expected_imus": [1, 2, 3]},
    10: {"code": "RKD", "name": "Right Knee Distal (Shank)", "role": "Core", "expected_imus": [1, 2, 3]},
}

CORE_DEVICE_IDS = [1, 2, 3, 4, 6, 7, 8, 10]
ALL_POTENTIAL_DEVICE_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
STANDARD_AXES = ["AX", "AY", "AZ", "GX", "GY", "GZ"]
TOTAL_EXPECTED_IMUS = sum(len(v["expected_imus"]) for v in TARGET_ANATOMICAL_MAP.values())

# Robust Regex supporting:
# - Device_1_IMU1_AX / Device_10_IMU3_GZ
# - OrbiX-Node-1_IMU1_AX / OrbiX-Node-10_IMU3_GZ
# - Node-1_IMU1_AX / Node_1_IMU1_AX / 1_IMU1_AX
# - Device_1_AX / device1_ax (defaults to IMU 1 if omitted)
COLUMN_REGEX = re.compile(
    r"(?:(?:OrbiX[_\-]Node|Device|Node)[_\-]?)?(\d+)(?:[_\-]IMU[_\-]?(\d+))?[_\-](AX|AY|AZ|GX|GY|GZ|MX|MY|MZ)$",
    re.IGNORECASE,
)

def format_bytes(size_bytes: int) -> str:
    """Format bytes into readable string (B, KB, MB, GB)."""
    if size_bytes <= 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    i = int(np.floor(np.log(size_bytes) / np.log(1024)))
    p = np.power(1024, i)
    s = round(size_bytes / p, 2)
    return f"{s} {units[i]}"

def format_duration(seconds: float) -> str:
    """Format seconds into HH:MM:SS or MM:SS."""
    if np.isnan(seconds) or seconds <= 0:
        return "00:00"
    m, s = divmod(int(round(seconds)), 60)
    h, m = divmod(m, 60)
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def df_to_markdown_safe(df: pd.DataFrame) -> str:
    """Generate Markdown table safely without requiring external tabulate."""
    try:
        return df.to_markdown(index=False)
    except Exception:
        headers = list(df.columns)
        rows = df.astype(str).values.tolist()
        col_widths = [
            max(len(str(h)), max((len(r[i]) for r in rows), default=0))
            for i, h in enumerate(headers)
        ]
        header_line = "| " + " | ".join(h.ljust(col_widths[i]) for i, h in enumerate(headers)) + " |"
        sep_line = "| " + " | ".join("-" * col_widths[i] for i in range(len(headers))) + " |"
        data_lines = [
            "| " + " | ".join(r[i].ljust(col_widths[i]) for i, h in enumerate(headers)) + " |"
            for r in rows
        ]
        return "\n".join([header_line, sep_line] + data_lines)

def parse_reps_from_notes(notes_str: str) -> int:
    """Extract repetition count from metadata notes, defaulting to 5 if standard."""
    if not isinstance(notes_str, str):
        return 5
    m = re.search(r"(\d+)\s*(?:reps|rep|repetition)", notes_str, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 5


class WearableBiomechanicsAuditor:
    def __init__(self, store, output_dir: str = "./dataset_paper_summary"):
        self.store = store
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        
        self.subjects_df = pd.DataFrame()
        self.experiments_df = pd.DataFrame()
        self.storage_stats = {}
        
        self.audit_results = []
        self.subject_summaries = []
        self.device_matrix_rows = []

    def run_full_audit(self) -> Dict[str, Any]:
        """Execute the complete audit pipeline."""
        start_time = time.time()
        print("\n" + "=" * 80)
        print("🏋🏽 WEARABLE BIOMECHANICS - DATASET PAPER AUDITOR")
        print("=" * 80)
        print(f"Base Directory : {self.store.base_dir}")
        print(f"Backend        : {self.store.backend}")
        print(f"Output Directory: {self.output_dir}")
        print("-" * 80)

        # 1. Storage Overview
        print("\n[Step 1/4] 💾 Analyzing Storage Volume & File Inventory...")
        self.audit_storage()

        # 2. Subject Cohort Analysis
        print("\n[Step 2/4] 👤 Profiling Registered Participant Cohort...")
        self.audit_subjects()

        # 3. Experiment Deep Audit
        print("\n[Step 3/4] 🧪 Auditing All Experimental Trials & Signal Integrity...")
        self.audit_experiments()

        # 4. Generate Summaries & Publication Artifacts
        print("\n[Step 4/4] 📄 Synthesizing Publication Deliverables & Reports...")
        final_deliverables = self.generate_paper_artifacts()

        elapsed = time.time() - start_time
        print("\n" + "=" * 80)
        print(f"✅ AUDIT COMPLETED IN {elapsed:.2f} SECONDS")
        print("=" * 80)
        return final_deliverables

    def audit_storage(self):
        """Analyze storage footprint, folder breakdown, and file types."""
        try:
            self.storage_stats = self.store.storage_summary(
                by_folder=True, depth=2, by_file_type=True, print_summary=False
            )
        except Exception as e:
            print(f"  ⚠️ Warning: storage_summary failed ({e}), using fallback.")
            self.storage_stats = {
                "total_size_bytes": 0,
                "total_size_formatted": "Unknown",
                "total_files": 0,
                "by_folder": {},
                "by_file_type": {},
            }

        total_size = self.storage_stats.get("total_size_formatted", "N/A")
        total_files = self.storage_stats.get("total_files", 0)
        print(f"  - Total Footprint: {total_size} ({total_files:,} files)")

    def audit_subjects(self):
        """Load and normalize subject registry data."""
        self.subjects_df = self.store.list_entities("subjects")
        if self.subjects_df.empty:
            print("  ⚠️ Warning: No subjects found in index!")
            return
        
        self.subjects_df.columns = [c.lower().replace(" ", "_") for c in self.subjects_df.columns]
        print(f"  - Total Registered Subjects: {len(self.subjects_df)}")

    def audit_experiments(self):
        """Iterate over all experiments and check raw CSVs, 10 devices, and all IMUs."""
        self.experiments_df = self.store.list_entries("experiments")
        if self.experiments_df.empty:
            print("  ⚠️ Warning: No experiments found in index!")
            return

        total_exps = len(self.experiments_df)
        print(f"  - Total Logged Sessions: {total_exps}")

        sub_lookup = {}
        if not self.subjects_df.empty:
            for _, srow in self.subjects_df.iterrows():
                sub_lookup[srow["entity_id"]] = srow.get("name", "Unknown")

        sub_session_stats = {
            sid: {"count": 0, "total_samples": 0, "total_duration": 0.0, "total_reps": 0}
            for sid in self.subjects_df["entity_id"].tolist()
        } if not self.subjects_df.empty else {}

        for idx, (_, exp_row) in enumerate(self.experiments_df.iterrows(), start=1):
            entry_id = exp_row["entry_id"]
            entity_id = exp_row.get("entity_id", "unknown")
            sub_name = sub_lookup.get(entity_id, "Unknown Subject")
            notes = str(exp_row.get("notes", "")).strip()
            activity_type = exp_row.get("description", "Clean & Jerk")
            weight_kg = exp_row.get("weight_kg", 20)
            date_str = str(exp_row.get("date", ""))
            time_str = str(exp_row.get("time", ""))
            ts_val = exp_row.get("timestamp", 0)
            reps_count = parse_reps_from_notes(notes)

            meta = self.store.get_entry("experiments", entry_id)
            related_dict = meta.get("related_files", {})
            proc_dict = meta.get("processed_files", {})

            # Related Files Checks
            has_zip = "raw_zip" in related_dict
            has_front = "front_video" in related_dict
            has_side = "side_video" in related_dict

            missing_related = []
            if not has_zip:
                missing_related.append("raw_zip")
            if not has_front:
                missing_related.append("front_video")
            if not has_side:
                missing_related.append("side_video")

            if has_front and has_side:
                video_status = "Complete (Front + Side)"
            elif has_side and not has_front:
                video_status = "Side View Only"
            elif has_front and not has_side:
                video_status = "Front View Only"
            else:
                video_status = "Missing All Videos"

            # Processed Files: Tags Only
            proc_tags_list = list(proc_dict.keys()) if isinstance(proc_dict, dict) else []
            proc_tags_str = ", ".join(proc_tags_list) if proc_tags_list else "None"

            # Raw IMU Data & Sensor Integrity Audit
            raw_path_rel = meta.get("raw_csv_path", "")
            raw_full_path = self.store._full_path(raw_path_rel)
            raw_exists = self.store.fs.exists(raw_full_path) if raw_path_rel else False

            raw_file_size_mb = 0.0
            row_count = 0
            duration_sec = 0.0
            sampling_rate_hz = 100.0
            total_nans = 0
            nan_pct = 0.0
            dead_channels = []
            detected_devices = set()
            device_imu_axes_map = {}
            missing_core_devices = []
            missing_axes_reports = []
            missing_imus_reports = []
            diagnostic_flags = []

            if raw_exists:
                try:
                    raw_file_size_mb = round(self.store.fs.size(raw_full_path) / (1024 * 1024), 2)
                except Exception:
                    raw_file_size_mb = 0.0

                try:
                    df_raw = self.store.get_entry_raw_data("experiments", entry_id)
                    row_count = len(df_raw)
                    total_cells = df_raw.size
                    total_nans = int(df_raw.isna().sum().sum())
                    nan_pct = round((total_nans / total_cells * 100), 2) if total_cells > 0 else 0.0

                    # Timestamp & duration estimation (supports 'Timestamp', 'stamp', 'time', etc.)
                    time_col_name = None
                    for c in df_raw.columns:
                        if c.lower() in ["timestamp", "stamp", "time"]:
                            time_col_name = c
                            break

                    if time_col_name:
                        ts_col = df_raw[time_col_name].dropna()
                        if len(ts_col) > 1:
                            dt_diff = ts_col.diff().dropna()
                            median_dt = dt_diff.median()
                            if median_dt > 0:
                                if median_dt > 5:  # likely milliseconds
                                    duration_sec = round((ts_col.iloc[-1] - ts_col.iloc[0]) / 1000.0, 2)
                                    sampling_rate_hz = round(1000.0 / median_dt, 1)
                                elif median_dt < 0.1:  # likely seconds
                                    duration_sec = round(ts_col.iloc[-1] - ts_col.iloc[0], 2)
                                    sampling_rate_hz = round(1.0 / median_dt, 1)
                                else:
                                    duration_sec = round(row_count / 100.0, 2)
                                    sampling_rate_hz = 100.0
                            else:
                                duration_sec = round(row_count / 100.0, 2)
                    else:
                        duration_sec = round(row_count / 100.0, 2)

                    # Sensor channel parsing across all naming conventions
                    for col in df_raw.columns:
                        if col == time_col_name or col.lower() in ["timestamp", "stamp", "time", "sample", "index"]:
                            continue
                        m = COLUMN_REGEX.search(col)
                        if m:
                            dev_id = int(m.group(1))
                            imu_id = int(m.group(2)) if m.group(2) else 1
                            axis_name = m.group(3).upper()

                            detected_devices.add(dev_id)
                            device_imu_axes_map.setdefault(dev_id, {}).setdefault(imu_id, set()).add(axis_name)

                        # Zero variance check (flatlined signal)
                        col_series = df_raw[col].dropna()
                        if len(col_series) > 1 and col_series.std() == 0.0:
                            dead_channels.append(col)

                    # Check for missing axes in detected IMUs
                    for dev_id, imus in device_imu_axes_map.items():
                        for imu_id, axes_set in imus.items():
                            missing_ax = set(STANDARD_AXES) - axes_set
                            if missing_ax:
                                missing_axes_reports.append(f"Dev{dev_id}_IMU{imu_id} missing {sorted(missing_ax)}")

                    # Check for missing core devices (1, 2, 3, 4, 6, 7, 8, 10)
                    missing_core_devices = [d for d in CORE_DEVICE_IDS if d not in detected_devices]

                    # Check for missing expected IMUs per device (Dev 1..4 have 2 IMUs; Dev 5..10 have 3 IMUs)
                    for d in ALL_POTENTIAL_DEVICE_IDS:
                        expected_imus = TARGET_ANATOMICAL_MAP[d]["expected_imus"]
                        dev_imus = device_imu_axes_map.get(d, {})
                        missing_for_dev = [imu for imu in expected_imus if imu not in dev_imus]
                        if missing_for_dev:
                            if d in detected_devices:
                                missing_imus_reports.append(f"Dev{d} missing IMUs {missing_for_dev}")
                            elif d in CORE_DEVICE_IDS:
                                missing_imus_reports.append(f"Dev{d} (Core) entirely missing ({len(expected_imus)} IMUs)")

                except Exception as err:
                    diagnostic_flags.append(f"Error reading CSV: {str(err)[:50]}")
            else:
                diagnostic_flags.append("Raw CSV missing from disk")

            # Anomaly Synthesis & Diagnostic Flags
            if missing_core_devices:
                diagnostic_flags.append(f"Missing Core Devices: {missing_core_devices}")
            if missing_imus_reports:
                diagnostic_flags.append(f"Missing IMUs: {', '.join(missing_imus_reports)}")
            if missing_axes_reports:
                diagnostic_flags.append(f"Incomplete Axes: {', '.join(missing_axes_reports)}")
            if dead_channels:
                diagnostic_flags.append(f"Flatline Channels ({len(dead_channels)} cols)")
            if nan_pct > 0.05:
                diagnostic_flags.append(f"NaN Rate: {nan_pct}%")
            if not has_front:
                diagnostic_flags.append("Missing Front Video")
            if not has_side:
                diagnostic_flags.append("Missing Side Video")
            if not has_zip:
                diagnostic_flags.append("Missing Raw Zip")
            if "detach" in notes.lower():
                diagnostic_flags.append("Sensor Detachment Noted")
            if "missing" in notes.lower() and not missing_core_devices:
                diagnostic_flags.append(f"Note Alert: {notes}")
            if not diagnostic_flags:
                diagnostic_flags.append("Clean / All Checks Passed")

            if entity_id in sub_session_stats:
                sub_session_stats[entity_id]["count"] += 1
                sub_session_stats[entity_id]["total_samples"] += row_count
                sub_session_stats[entity_id]["total_duration"] += duration_sec
                sub_session_stats[entity_id]["total_reps"] += reps_count

            total_imus_detected = sum(len(imus) for imus in device_imu_axes_map.values())

            status_symbol = "✓" if "All Checks Passed" in diagnostic_flags or len(diagnostic_flags) == 0 else "⚠️"
            print(f"  [{idx:02d}/{total_exps:02d}] {entry_id} ({entity_id}) -> {row_count:,} rows, {len(detected_devices)}/10 devs, {total_imus_detected}/{TOTAL_EXPECTED_IMUS} IMUs, {video_status} {status_symbol}")

            self.audit_results.append({
                "entry_id": entry_id,
                "entity_id": entity_id,
                "subject_name": sub_name,
                "date": date_str,
                "time": time_str,
                "timestamp": ts_val,
                "activity_type": activity_type,
                "weight_kg": weight_kg,
                "reps": reps_count,
                "raw_file_name": os.path.basename(raw_path_rel) if raw_path_rel else "N/A",
                "raw_file_exists": raw_exists,
                "raw_file_size_mb": raw_file_size_mb,
                "row_count": row_count,
                "duration_sec": duration_sec,
                "duration_fmt": format_duration(duration_sec),
                "sampling_rate_hz": sampling_rate_hz,
                "total_nans": total_nans,
                "nan_pct": nan_pct,
                "dead_channels_count": len(dead_channels),
                "devices_detected_count": len(detected_devices),
                "total_imus_detected": total_imus_detected,
                "devices_detected_list": sorted(list(detected_devices)),
                "missing_core_devices": missing_core_devices if missing_core_devices else "None",
                "missing_imus": "; ".join(missing_imus_reports) if missing_imus_reports else "None",
                "missing_axes": "; ".join(missing_axes_reports) if missing_axes_reports else "None",
                "has_raw_zip": "Yes" if has_zip else "No",
                "has_front_video": "Yes" if has_front else "No",
                "has_side_video": "Yes" if has_side else "No",
                "video_status": video_status,
                "missing_related_files": ", ".join(missing_related) if missing_related else "None",
                "processed_tags": proc_tags_str,
                "researcher_notes": notes if notes else "None",
                "diagnostic_flags": " | ".join(diagnostic_flags),
            })

            # Record device matrix row (Experiment vs Devices 1..10 overview + all 26 individual IMU nodes)
            dev_row = {"entry_id": entry_id, "entity_id": entity_id}
            
            # 1. Device overview columns
            for d in ALL_POTENTIAL_DEVICE_IDS:
                info = TARGET_ANATOMICAL_MAP[d]
                code = info["code"]
                expected = info["expected_imus"]
                col_key = f"Dev_{d}_{code}"
                if d in detected_devices:
                    dev_imus = device_imu_axes_map.get(d, {})
                    present = [i for i in expected if i in dev_imus]
                    all_axes_ok = all(len(dev_imus[i] & set(STANDARD_AXES)) == 6 for i in present)
                    if len(present) == len(expected) and all_axes_ok:
                        dev_row[col_key] = f"✓ ({len(expected)} IMUs)"
                    elif len(present) > 0:
                        dev_row[col_key] = f"⚠️ ({len(present)}/{len(expected)} IMUs)"
                    else:
                        dev_row[col_key] = "✗"
                else:
                    dev_row[col_key] = "✗ (Missing)" if d in CORE_DEVICE_IDS else "-"

            # 2. Granular per-IMU columns (all 26 individual nodes across the body)
            for d in ALL_POTENTIAL_DEVICE_IDS:
                for imu in TARGET_ANATOMICAL_MAP[d]["expected_imus"]:
                    k = f"D{d}_IMU{imu}"
                    if imu in device_imu_axes_map.get(d, {}):
                        axes = device_imu_axes_map[d][imu]
                        dev_row[k] = "✓" if len(axes & set(STANDARD_AXES)) == 6 else "⚠️"
                    else:
                        dev_row[k] = "✗"
            self.device_matrix_rows.append(dev_row)

        for _, srow in self.subjects_df.iterrows():
            sid = srow["entity_id"]
            name = srow.get("name", "Unknown")
            age = srow.get("age", 0)
            weight = srow.get("weight", 0)
            height = srow.get("height", 0.0)
            experience = srow.get("experience", 0)
            achievement = srow.get("achievement", "-")
            best_weight = srow.get("best_weight", "-")
            notes = srow.get("notes", "-")

            h_val = float(height) if height else 0.0
            if h_val > 3.0:
                h_val = round(h_val / 100.0, 2)

            w_val = float(weight) if weight else 0.0
            bmi = round(w_val / (h_val ** 2), 1) if (h_val > 0.5 and w_val > 10) else 0.0

            stats = sub_session_stats.get(sid, {"count": 0, "total_samples": 0, "total_duration": 0.0, "total_reps": 0})
            is_placeholder = (age == 0 and w_val == 0)

            self.subject_summaries.append({
                "entity_id": sid,
                "name": name,
                "age": int(age) if age else 0,
                "weight_kg": w_val,
                "height_m": h_val,
                "bmi": bmi if bmi > 0 else "-",
                "experience_years": int(experience) if experience else 0,
                "achievement": achievement,
                "best_weight_kg": best_weight,
                "sports_background": notes,
                "total_sessions": stats["count"],
                "total_reps": stats["total_reps"],
                "total_duration_sec": stats["total_duration"],
                "total_duration_fmt": format_duration(stats["total_duration"]),
                "total_samples": stats["total_samples"],
                "metadata_status": "⚠️ Incomplete/Placeholder" if is_placeholder else "✓ Complete",
            })

    def generate_paper_artifacts(self) -> Dict[str, Any]:
        """Synthesize all tables into CSVs, JSON, and a comprehensive Markdown paper companion."""
        df_audit = pd.DataFrame(self.audit_results)
        df_sub_sum = pd.DataFrame(self.subject_summaries)
        df_matrix = pd.DataFrame(self.device_matrix_rows)

        # 1. Save CSVs
        csv_audit_path = os.path.join(self.output_dir, "experiments_detailed_audit.csv")
        csv_sub_path = os.path.join(self.output_dir, "subjects_summary.csv")
        csv_mat_path = os.path.join(self.output_dir, "sensor_completeness_matrix.csv")

        df_audit.to_csv(csv_audit_path, index=False)
        df_sub_sum.to_csv(csv_sub_path, index=False)
        df_matrix.to_csv(csv_mat_path, index=False)

        # 2. Compute Publication Metrics
        total_sessions = len(df_audit)
        total_samples = int(df_audit["row_count"].sum()) if not df_audit.empty else 0
        total_duration_s = float(df_audit["duration_sec"].sum()) if not df_audit.empty else 0.0
        total_reps = int(df_audit["reps"].sum()) if not df_audit.empty else 0
        
        valid_subs = df_sub_sum[df_sub_sum["age"] > 0]
        age_mean = f"{valid_subs['age'].mean():.1f} ± {valid_subs['age'].std():.1f}" if len(valid_subs) > 1 else "N/A"
        age_range = f"[{valid_subs['age'].min()} - {valid_subs['age'].max()}]" if len(valid_subs) > 0 else "N/A"
        
        valid_w = valid_subs[valid_subs["weight_kg"] > 0]
        weight_mean = f"{valid_w['weight_kg'].mean():.1f} ± {valid_w['weight_kg'].std():.1f}" if len(valid_w) > 1 else "N/A"
        weight_range = f"[{valid_w['weight_kg'].min()} - {valid_w['weight_kg'].max()}]" if len(valid_w) > 0 else "N/A"

        valid_h = valid_subs[valid_subs["height_m"] > 0]
        height_mean = f"{valid_h['height_m'].mean():.2f} ± {valid_h['height_m'].std():.2f}" if len(valid_h) > 1 else "N/A"
        height_range = f"[{valid_h['height_m'].min()} - {valid_h['height_m'].max()}]" if len(valid_h) > 0 else "N/A"

        dual_video_count = sum(df_audit["video_status"] == "Complete (Front + Side)")
        side_only_count = sum(df_audit["video_status"] == "Side View Only")
        front_only_count = sum(df_audit["video_status"] == "Front View Only")
        zip_count = sum(df_audit["has_raw_zip"] == "Yes")

        dev6_missing_count = sum(df_audit["missing_core_devices"].astype(str).str.contains("6"))
        sensor_detach_count = sum(df_audit["diagnostic_flags"].astype(str).str.contains("Sensor Detachment"))

        st = self.storage_stats
        total_storage_gb = st.get("total_size_formatted", "29.35 GB")

        # 3. Save JSON stats
        json_stats_path = os.path.join(self.output_dir, "dataset_stats.json")
        stats_payload = {
            "dataset_title": "Wearable Biomechanics Dataset: Multi-Modal Synchronized Kinematics for Olympic Weightlifting",
            "audit_timestamp": datetime.now().isoformat(),
            "summary_metrics": {
                "total_registered_subjects": len(df_sub_sum),
                "active_recording_subjects": len(df_sub_sum[df_sub_sum["total_sessions"] > 0]),
                "total_experimental_sessions": total_sessions,
                "total_repetitions": total_reps,
                "total_recorded_duration_seconds": total_duration_s,
                "total_recorded_duration_formatted": format_duration(total_duration_s),
                "total_imu_samples": total_samples,
                "nominal_imu_frequency_hz": 100,
                "nominal_video_fps": 30,
                "total_storage_footprint": total_storage_gb,
            },
            "cohort_anthropometrics": {
                "age_years": {"mean_sd": age_mean, "range": age_range, "n_reported": len(valid_subs)},
                "weight_kg": {"mean_sd": weight_mean, "range": weight_range, "n_reported": len(valid_w)},
                "height_m": {"mean_sd": height_mean, "range": height_range, "n_reported": len(valid_h)},
            },
            "hardware_sensor_architecture": {
                "total_devices": 10,
                "nominal_imu_nodes_total": TOTAL_EXPECTED_IMUS,
                "expected_imus_breakdown": {
                    f"device_{d}": TARGET_ANATOMICAL_MAP[d]["expected_imus"]
                    for d in ALL_POTENTIAL_DEVICE_IDS
                }
            },
            "multi_modal_inventory": {
                "dual_camera_sessions": dual_video_count,
                "single_camera_side_only_sessions": side_only_count,
                "single_camera_front_only_sessions": front_only_count,
                "raw_device_zip_archives": zip_count,
            },
            "technical_validation_anomalies": {
                "device_6_missing_sessions": dev6_missing_count,
                "sensor_detachment_sessions": sensor_detach_count,
            }
        }
        with open(json_stats_path, "w") as jf:
            json.dump(stats_payload, jf, indent=2)

        # 4. Generate Comprehensive Publication-Ready Markdown Report
        md_path = os.path.join(self.output_dir, "dataset_summary_for_paper.md")
        with open(md_path, "w") as mf:
            mf.write("# 🏋🏽 Dataset Paper Companion: Wearable Biomechanics\n\n")
            mf.write("> **Prepared for:** Dataset Descriptor Publication (*Nature Scientific Data* / *Elsevier Data in Brief* / *Sports Biomechanics*)\n")
            mf.write(f"> **Generated on:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | **Repository:** `Wearable-Biomechanics`\n\n")

            mf.write("## 1. Executive Summary & Dataset Inventory\n\n")
            mf.write("| Metric | Quantified Value | Description |\n")
            mf.write("| :--- | :--- | :--- |\n")
            mf.write(f"| **Total Subjects** | **{len(df_sub_sum)}** athletes | 13 registered participants across novice, intermediate, and advanced lifters |\n")
            mf.write(f"| **Total Sessions** | **{total_sessions}** trials | Continuous multi-rep experimental sessions (Clean & Jerk) |\n")
            mf.write(f"| **Total Repetitions** | **{total_reps}** reps | Standard 5 reps per recording session |\n")
            mf.write(f"| **Total IMU Samples** | **{total_samples:,}** samples | Multi-sensor time-series at 100 Hz |\n")
            mf.write(f"| **Total Duration** | **{format_duration(total_duration_s)}** (MM:SS) | Cumulative recorded kinematics time |\n")
            mf.write(f"| **Total Storage Size** | **{total_storage_gb}** | Multi-modal bundle across raw, processed, and video files |\n")
            mf.write(f"| **Nominal IMU Rate** | **100 Hz** | Synchronized 6-DoF accelerometer + gyroscope telemetry |\n")
            mf.write(f"| **Video Capture Rate** | **30 fps** | Multi-view synchronized Front & Side RGB video captures |\n\n")

            mf.write("## 2. Participant Cohort & Anthropometrics (Table 1 for Paper)\n\n")
            mf.write(f"**Cohort Demographics Summary (Reported N={len(valid_subs)}):**\n")
            mf.write(f"- **Age:** {age_mean} years {age_range}\n")
            mf.write(f"- **Body Mass:** {weight_mean} kg {weight_range}\n")
            mf.write(f"- **Stature:** {height_mean} m {height_range}\n")
            mf.write(f"- **Lifter Experience:** 0 to 4 years (Competitive medaled athletes to novices)\n")
            mf.write("- **Gender Distribution:** Predominantly male athletes with female lifter representation (`sub_0005`, 2 trials, 10 reps).\n")
            mf.write(f"- *Note on Incomplete Questionnaires:* {len(df_sub_sum) - len(valid_subs)} subjects (`sub_0004` to `sub_0010`, `sub_0013`) currently have placeholder demographic parameters awaiting paper finalization.\n\n")

            sub_display = df_sub_sum[[
                "entity_id", "name", "age", "weight_kg", "height_m", "experience_years",
                "achievement", "sports_background", "total_sessions", "total_reps", "metadata_status"
            ]]
            mf.write(df_to_markdown_safe(sub_display) + "\n\n")

            mf.write("## 3. Storage Hierarchy & Modality Breakdown\n\n")
            if "by_folder" in st and st["by_folder"]:
                mf.write("### Folder Directory Breakdown\n\n")
                folder_records = [
                    {"Folder": k, "Size": v["size_formatted"], "File Count": v["file_count"], "Percentage": f"{v['percentage']:.1f}%"}
                    for k, v in st["by_folder"].items()
                ]
                mf.write(df_to_markdown_safe(pd.DataFrame(folder_records)) + "\n\n")

            if "by_file_type" in st and st["by_file_type"]:
                mf.write("### File Type Breakdown\n\n")
                type_records = [
                    {"File Type": k, "Size": v["size_formatted"], "File Count": v["file_count"], "Percentage": f"{v['percentage']:.1f}%"}
                    for k, v in st["by_file_type"].items()
                ]
                mf.write(df_to_markdown_safe(pd.DataFrame(type_records)) + "\n\n")

            mf.write("## 4. Hardware Sensor Setup & Multi-IMU Node Completeness\n\n")
            mf.write("The experimental body-area sensor network instruments the anatomical joints as follows:\n\n")
            mf.write("| Device ID | Code | Anatomical Placement | Primary Role | Expected IMUs | Evaluated Axes |\n")
            mf.write("| :---: | :---: | :--- | :---: | :---: | :--- |\n")
            for dev_id, info in TARGET_ANATOMICAL_MAP.items():
                imus_str = f"{len(info['expected_imus'])} (IMUs {', '.join(str(i) for i in info['expected_imus'])})"
                mf.write(f"| Device {dev_id} | `{info['code']}` | {info['name']} | {info['role']} | {imus_str} | AX, AY, AZ, GX, GY, GZ |\n")
            mf.write("\n\n")

            # Availability rate per device and per IMU
            mf.write("### Sensor & IMU Availability Rate Across All Sessions\n\n")
            dev_avail_summary = []
            for d in ALL_POTENTIAL_DEVICE_IDS:
                info = TARGET_ANATOMICAL_MAP[d]
                code = info["code"]
                expected = info["expected_imus"]
                col_key = f"Dev_{d}_{code}"
                
                present_cnt = sum(df_matrix[col_key].astype(str).str.contains("✓")) if col_key in df_matrix.columns else 0
                partial_cnt = sum(df_matrix[col_key].astype(str).str.contains("⚠️")) if col_key in df_matrix.columns else 0
                avail_pct = round(((present_cnt + partial_cnt) / total_sessions * 100), 1) if total_sessions > 0 else 0.0

                imu_statuses = []
                for imu in expected:
                    imu_col = f"D{d}_IMU{imu}"
                    if imu_col in df_matrix.columns:
                        imu_ok = sum(df_matrix[imu_col] == "✓")
                        imu_statuses.append(f"IMU{imu}: {round(imu_ok / total_sessions * 100, 1)}%")
                    else:
                        imu_statuses.append(f"IMU{imu}: N/A")

                dev_avail_summary.append({
                    "Device ID": f"Device {d}",
                    "Anatomical Code": code,
                    "Placement": info["name"],
                    "Role": info["role"],
                    "Expected Nodes": f"{len(expected)} IMUs",
                    "Per-IMU Availability": "; ".join(imu_statuses),
                    "Full Device Health": f"{avail_pct}%",
                })
            mf.write(df_to_markdown_safe(pd.DataFrame(dev_avail_summary)) + "\n\n")

            mf.write("## 5. Multi-View Video & Attachment Completeness\n\n")
            mf.write(f"- **Dual-Camera Synchronized Sessions (Front + Side):** {dual_video_count} / {total_sessions} ({round(dual_video_count / total_sessions * 100, 1)}%)\n")
            mf.write(f"- **Single-Camera Sessions (Side Only):** {side_only_count} / {total_sessions} (Trials `experiments_0011` through `experiments_0023`)\n")
            mf.write(f"- **Single-Camera Sessions (Front Only):** {front_only_count} / {total_sessions} (Trial `experiments_0007`)\n")
            mf.write(f"- **Raw Device Archives (.zip):** {zip_count} / {total_sessions} (100% complete across all sessions)\n\n")

            mf.write("## 6. Technical Validation & Known Experimental Anomalies\n\n")
            mf.write("For complete scientific transparency, the following experimental deviations are cataloged:\n\n")
            mf.write("1. **Sensor Detachments:**\n")
            mf.write("   - `experiments_0001` (sub_0001): Left ankle sensor detached during the last 3 reps.\n")
            mf.write("   - `experiments_0002` (sub_0001): Left ankle sensor detached during rep 3.\n\n")
            mf.write("2. **Hardware Exclusions (Device 6 Missing):**\n")
            mf.write("   - `experiments_0003`, `experiments_0004`, `experiments_0005`: Device 6 (Right Elbow Distal / Forearm & Wrist) experienced telemetry disruption and was excluded.\n\n")
            mf.write("3. **Missing Front Perspective Videos:**\n")
            mf.write("   - Experiments `0011` to `0023` were collected with a single high-speed lateral (Side) camera only.\n\n")

            mf.write("## 7. Master Experiment Audit Table\n\n")
            mf.write("Below is the exhaustive, session-by-session inventory.\n\n")
            audit_display_cols = [
                "entry_id", "entity_id", "subject_name", "date", "weight_kg", "reps",
                "row_count", "duration_fmt", "devices_detected_count", "total_imus_detected",
                "video_status", "missing_core_devices", "missing_imus", "diagnostic_flags"
            ]
            mf.write(df_to_markdown_safe(df_audit[audit_display_cols]) + "\n\n")

        # 5. Create Zip Bundle
        bundle_zip_path = os.path.join(self.output_dir, "dataset_paper_bundle.zip")
        with zipfile.ZipFile(bundle_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.write(md_path, arcname="dataset_summary_for_paper.md")
            zf.write(csv_audit_path, arcname="experiments_detailed_audit.csv")
            zf.write(csv_sub_path, arcname="subjects_summary.csv")
            zf.write(csv_mat_path, arcname="sensor_completeness_matrix.csv")
            zf.write(json_stats_path, arcname="dataset_stats.json")

        print(f"\n🎉 Saved Master Paper Summary Report -> {md_path}")
        print(f"📊 Saved Detailed Experiment Audit CSV -> {csv_audit_path}")
        print(f"👤 Saved Subject Cohort Summary CSV  -> {csv_sub_path}")
        print(f"🧩 Saved Sensor Completeness Matrix  -> {csv_mat_path}")
        print(f"📦 Saved 1-Click Download Zip Bundle -> {bundle_zip_path}")

        return {
            "markdown_report": md_path,
            "audit_csv": csv_audit_path,
            "subjects_csv": csv_sub_path,
            "matrix_csv": csv_mat_path,
            "stats_json": json_stats_path,
            "zip_bundle": bundle_zip_path,
            "df_audit": df_audit,
            "df_sub_sum": df_sub_sum,
            "df_matrix": df_matrix,
            "stats_payload": stats_payload,
            "total_sessions": total_sessions,
            "total_samples": total_samples,
            "total_duration_sec": total_duration_s,
        }

print("✅ WearableBiomechanicsAuditor ready.")

✅ WearableBiomechanicsAuditor ready.


## **🚀 4. Execute Complete Dataset Audit Pipeline**
Run the deep verification across all participant records, trial time-series, video assets, and hardware channels.

In [5]:
# @title 🧪 Run Full Dataset Audit Pipeline { display-mode: "form" }
# @markdown Run the complete profiling and save deliverables to disk and Google Drive.

auto_download_bundle = True # @param {type:"boolean"}
copy_to_google_drive = True # @param {type:"boolean"}
drive_backup_subfolder = "summary_reports" # @param {type:"string"}

auditor = WearableBiomechanicsAuditor(store=store, output_dir=output_directory)
deliverables = auditor.run_full_audit()

# Backup deliverables to Google Drive if requested
if copy_to_google_drive and IN_COLAB:
    drive_backup_dir = os.path.join(dataset_path, drive_backup_subfolder)
    os.makedirs(drive_backup_dir, exist_ok=True)
    for k in ["markdown_report", "audit_csv", "subjects_csv", "matrix_csv", "stats_json", "zip_bundle"]:
        src_f = deliverables[k]
        dst_f = os.path.join(drive_backup_dir, os.path.basename(src_f))
        shutil.copy2(src_f, dst_f)
    print(f"\n💾 Permanent backup successfully saved to Google Drive: {drive_backup_dir}")

# Colab automatic browser download
if auto_download_bundle and IN_COLAB:
    print("\n📥 Triggering automatic browser download of dataset_paper_bundle.zip...")
    try:
        files.download(deliverables["zip_bundle"])
        print("✅ Download initiated in browser.")
    except Exception as e:
        print(f"  Note: Automatic browser download prompt bypassed ({e}). Available at: {deliverables['zip_bundle']}")


🏋🏽 WEARABLE BIOMECHANICS - DATASET PAPER AUDITOR
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
Output Directory: ./dataset_paper_summary
--------------------------------------------------------------------------------

[Step 1/4] 💾 Analyzing Storage Volume & File Inventory...
  - Total Footprint: 29.35 GB (293 files)

[Step 2/4] 👤 Profiling Registered Participant Cohort...
  - Total Registered Subjects: 13

[Step 3/4] 🧪 Auditing All Experimental Trials & Signal Integrity...
  - Total Logged Sessions: 38
  [01/38] experiments_0001 (sub_0001) -> 7,713 rows, 10/10 devs, 26/26 IMUs, Complete (Front + Side) ⚠️
  [02/38] experiments_0002 (sub_0001) -> 7,282 rows, 10/10 devs, 26/26 IMUs, Complete (Front + Side) ⚠️
  [03/38] experiments_0003 (sub_0002) -> 10,263 rows, 10/10 devs, 26/26 IMUs, Complete (Front + Side) ⚠️
  [04/38] experiments_0004 (sub_0002) -> 10,263 rows, 10/10 devs, 26/26 IMUs, Complete (Front + Side) ⚠️
  [05/38] experiments_00

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download initiated in browser.


## **📊 5. Publication Dashboards & Visual Profiling**
Interactive visual summaries for quick inspection of cohort characteristics, multi-IMU sensor availability, video coverage, and trial status.

In [6]:
# @title 🏆 Executive Summary & KPI Metrics { display-mode: "form" }
# @markdown Visual presentation of dataset key performance indicators.

from IPython.display import HTML, display

stats = deliverables["stats_payload"]["summary_metrics"]
anthro = deliverables["stats_payload"]["cohort_anthropometrics"]
video_stats = deliverables["stats_payload"]["multi_modal_inventory"]

kpi_html = f"""
<div style="font-family: Arial, sans-serif; padding: 15px; background: #0f172a; border-radius: 12px; color: #f8fafc; margin-bottom: 20px;">
  <h2 style="margin-top: 0; color: #38bdf8; border-bottom: 1px solid #334155; padding-bottom: 8px;">
    🏋🏽 Wearable Biomechanics Dataset Metrics
  </h2>
  <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-top: 15px;">
    <div style="background: #1e293b; padding: 12px; border-radius: 8px; border-left: 4px solid #38bdf8;">
      <div style="color: #94a3b8; font-size: 12px; font-weight: bold; text-transform: uppercase;">Participants</div>
      <div style="font-size: 24px; font-weight: bold; color: #ffffff; margin-top: 4px;">{stats['total_registered_subjects']} <span style="font-size: 13px; color: #38bdf8;">({stats['active_recording_subjects']} Active)</span></div>
      <div style="font-size: 11px; color: #cbd5e1; margin-top: 4px;">Age: {anthro['age_years']['mean_sd']} yrs</div>
    </div>
    <div style="background: #1e293b; padding: 12px; border-radius: 8px; border-left: 4px solid #34d399;">
      <div style="color: #94a3b8; font-size: 12px; font-weight: bold; text-transform: uppercase;">Experimental Trials</div>
      <div style="font-size: 24px; font-weight: bold; color: #ffffff; margin-top: 4px;">{stats['total_experimental_sessions']}</div>
      <div style="font-size: 11px; color: #cbd5e1; margin-top: 4px;">Total: {stats['total_repetitions']} repetitions</div>
    </div>
    <div style="background: #1e293b; padding: 12px; border-radius: 8px; border-left: 4px solid #f472b6;">
      <div style="color: #94a3b8; font-size: 12px; font-weight: bold; text-transform: uppercase;">Kinematic Telemetry</div>
      <div style="font-size: 24px; font-weight: bold; color: #ffffff; margin-top: 4px;">{stats['total_imu_samples']:,}</div>
      <div style="font-size: 11px; color: #cbd5e1; margin-top: 4px;">100 Hz | 26 IMU Nodes ({stats['total_recorded_duration_formatted']})</div>
    </div>
    <div style="background: #1e293b; padding: 12px; border-radius: 8px; border-left: 4px solid #fbbf24;">
      <div style="color: #94a3b8; font-size: 12px; font-weight: bold; text-transform: uppercase;">Storage Footprint</div>
      <div style="font-size: 24px; font-weight: bold; color: #ffffff; margin-top: 4px;">{stats['total_storage_footprint']}</div>
      <div style="font-size: 11px; color: #cbd5e1; margin-top: 4px;">Raw, Parquet & Dual-View Video</div>
    </div>
  </div>
</div>
"""
display(HTML(kpi_html))

In [7]:
# @title 👤 Participant Cohort Demographics Table { display-mode: "form" }
# @markdown View subject profile details, lifting experience, and recorded session volume.

filter_status = "All" # @param ["All", "Complete Only", "Placeholder Only"]

df_sub_display = deliverables["df_sub_sum"].copy()

if filter_status == "Complete Only":
    df_sub_display = df_sub_display[df_sub_display["metadata_status"].str.contains("Complete")]
elif filter_status == "Placeholder Only":
    df_sub_display = df_sub_display[df_sub_display["metadata_status"].str.contains("Placeholder")]

cols_to_show = [
    "entity_id", "name", "age", "weight_kg", "height_m", "bmi",
    "experience_years", "achievement", "sports_background", "total_sessions", "total_reps", "metadata_status"
]

try:
    from google.colab import data_table
    display(data_table.DataTable(df_sub_display[cols_to_show], include_index=False, num_rows_per_page=15))
except Exception:
    display(df_sub_display[cols_to_show])

,entity_id,name,age,weight_kg,height_m,bmi,experience_years,achievement,sports_background,total_sessions,total_reps,metadata_status
0,sub_0001,Hivindu Kirulapana,22,72.0,1.74,23.8,1,-,Participation in Other Sports - Rugby,2,10,✓ Complete
1,sub_0002,Yasith Dissanayake,22,69.0,1.78,21.8,4,-,Participation in Other Sports - Rugby,5,25,✓ Complete
2,sub_0003,Yewin Dissanayake,22,65.0,1.70,22.5,2,Intermediate bronze medal 2025,Participation in Other sports: none,5,25,✓ Complete
3,sub_0004,Mahasen,0,0.0,0.00,-,0,N\A,N\A,4,20,⚠️ Incomplete/Placeholder
4,sub_0005,Achalani,0,0.0,0.00,-,0,N\A,N\A,2,10,⚠️ Incomplete/Placeholder
5,sub_0006,Pandula,0,0.0,0.00,-,0,NA,NA,3,15,⚠️ Incomplete/Placeholder
6,sub_0007,Enuka,0,0.0,0.00,-,0,NA,NA,2,10,⚠️ Incomplete/Placeholder
7,sub_0008,Thevindu,0,0.0,0.00,-,0,NA,NA,5,21,⚠️ Incomplete/Placeholder
8,sub_0009,Induwara,0,0.0,0.00,-,0,NA,NA,2,10,⚠️ Incomplete/Placeholder
9,sub_0010,Duleetha,0,0.0,0.00,-,0,NA,NA,2,10,⚠️ Incomplete/Placeholder


In [8]:
# @title 🧩 Sensor & IMU Completeness Matrix (Heatmap) { display-mode: "form" }
# @markdown Visualizes all 26 body IMU nodes (or 10-device overview) across every experimental trial.

view_mode = "All 26 IMU Nodes" # @param ["All 26 IMU Nodes", "Device Overview"]

df_matrix_viz = deliverables["df_matrix"].set_index("entry_id").drop(columns=["entity_id"])

if view_mode == "All 26 IMU Nodes":
    cols_to_plot = [c for c in df_matrix_viz.columns if "_IMU" in c]
    title_text = "Granular 26-Node Body IMU Completeness Matrix (Green = 6-DoF Validated, Red = Missing)"
else:
    cols_to_plot = [c for c in df_matrix_viz.columns if c.startswith("Dev_")]
    title_text = "10-Device Anatomical Kinematic Chain Overview Matrix"

num_matrix = df_matrix_viz[cols_to_plot].copy()

def map_status_to_num(val):
    if not isinstance(val, str):
        return np.nan
    if "✓" in val:
        return 1.0
    elif "⚠️" in val:
        return 0.5
    elif "✗" in val:
        return 0.0
    return np.nan

num_matrix = num_matrix.applymap(map_status_to_num).astype(float)

fig = px.imshow(
    num_matrix,
    labels=dict(x="Sensor / IMU Node", y="Experimental Session", color="Status"),
    x=num_matrix.columns,
    y=num_matrix.index,
    color_continuous_scale=[
        [0.0, "#ef4444"],   # Red: Missing
        [0.5, "#f59e0b"],   # Orange: Degraded / Partial Axes
        [1.0, "#10b981"]    # Green: Complete 6-DoF
    ],
    aspect="auto",
    title=title_text
)

fig.update_layout(
    height=max(500, len(num_matrix) * 16),
    template="simple_white",
    xaxis=dict(tickangle=-45),
    coloraxis_colorbar=dict(
        tickvals=[0.0, 0.5, 1.0],
        ticktext=["Missing (✗)", "Degraded (⚠️)", "Complete (✓)"]
    )
)

fig.show()

/tmp/ipykernel_972/3965431327.py:28: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



In [9]:
# @title 🎥 Multi-Camera Coverage & Modality Distribution { display-mode: "form" }
# @markdown Visual distribution of video perspectives (Dual-Camera vs Lateral Only vs Front Only).

df_aud = deliverables["df_audit"]
video_counts = df_aud["video_status"].value_counts().reset_index()
video_counts.columns = ["Video Perspective", "Count"]

fig_video = px.pie(
    video_counts,
    names="Video Perspective",
    values="Count",
    title="Synchronized Video Coverage Distribution across Trials",
    color="Video Perspective",
    color_discrete_map={
        "Complete (Front + Side)": "#10b981",
        "Side View Only": "#3b82f6",
        "Front View Only": "#f59e0b",
        "Missing All Videos": "#ef4444"
    },
    hole=0.45
)
fig_video.update_traces(textinfo='percent+label')
fig_video.update_layout(template="simple_white")
fig_video.show()

In [10]:
# @title 📋 Full Experiment Audit Explorer { display-mode: "form" }
# @markdown Search, filter, and inspect trial metrics, durations, detected devices, IMU counts, and researcher anomaly flags.

audit_cols = [
    "entry_id", "entity_id", "subject_name", "date", "weight_kg", "reps",
    "row_count", "duration_fmt", "sampling_rate_hz", "nan_pct",
    "devices_detected_count", "total_imus_detected", "missing_core_devices",
    "missing_imus", "video_status", "diagnostic_flags"
]

try:
    from google.colab import data_table
    display(data_table.DataTable(deliverables["df_audit"][audit_cols], include_index=False, num_rows_per_page=15))
except Exception:
    display(deliverables["df_audit"][audit_cols])

,entry_id,entity_id,subject_name,date,weight_kg,reps,row_count,duration_fmt,sampling_rate_hz,nan_pct,devices_detected_count,total_imus_detected,missing_core_devices,missing_imus,video_status,diagnostic_flags
0,experiments_0001,sub_0001,Hivindu Kirulapana,2026-05-08,20,5,7713,01:17,100.0,0.00,10,26,None,None,Complete (Front + Side),Sensor Detachment Noted
1,experiments_0002,sub_0001,Hivindu Kirulapana,2026-05-08,20,5,7282,01:13,100.0,0.00,10,26,None,None,Complete (Front + Side),Sensor Detachment Noted
2,experiments_0003,sub_0002,Yasith Dissanayake,2026-05-13,20,5,10263,01:43,100.0,0.00,10,26,None,None,Complete (Front + Side),Flatline Channels (18 cols) | Note Alert: 5 re...
3,experiments_0004,sub_0002,Yasith Dissanayake,2026-05-13,20,5,10263,01:43,100.0,0.00,10,26,None,None,Complete (Front + Side),Flatline Channels (18 cols) | Note Alert: 5 re...
4,experiments_0005,sub_0003,Yewin Dissanayake,2026-05-13,20,5,10263,01:43,100.0,0.00,10,26,None,None,Complete (Front + Side),Flatline Channels (18 cols) | Note Alert: 5 re...
5,experiments_0006,sub_0003,Yewin Dissanayake,2026-05-13,20,5,10263,01:43,100.0,0.00,10,26,None,None,Complete (Front + Side),Flatline Channels (18 cols)
6,experiments_0007,sub_0004,Mahasen,2026-06-10,20,5,11108,01:51,100.0,0.00,10,26,None,None,Front View Only,Missing Side Video
7,experiments_0008,sub_0004,Mahasen,2026-06-10,20,5,7113,01:11,100.0,0.00,10,26,None,None,Complete (Front + Side),Clean / All Checks Passed
8,experiments_0009,sub_0004,Mahasen,2026-06-10,20,5,5027,00:50,100.0,0.00,10,26,None,None,Complete (Front + Side),Clean / All Checks Passed
9,experiments_0010,sub_0004,Mahasen,2026-06-10,20,5,5182,00:52,100.0,0.00,10,26,None,None,Complete (Front + Side),Clean / All Checks Passed


## **📦 6. Export & Publication Deliverables**
Download individual audit files or preview the synthesized master Markdown paper companion directly.

In [11]:
# @title 📄 Preview Generated Paper Markdown Report { display-mode: "form" }
# @markdown Render the generated publication descriptor directly inside the notebook.

from IPython.display import Markdown

with open(deliverables["markdown_report"], "r") as f:
    report_text = f.read()

display(Markdown(report_text))

# 🏋🏽 Dataset Paper Companion: Wearable Biomechanics

> **Prepared for:** Dataset Descriptor Publication (*Nature Scientific Data* / *Elsevier Data in Brief* / *Sports Biomechanics*)
> **Generated on:** 2026-10-14 17:05:55 | **Repository:** `Wearable-Biomechanics`

## 1. Executive Summary & Dataset Inventory

| Metric | Quantified Value | Description |
| :--- | :--- | :--- |
| **Total Subjects** | **13** athletes | 13 registered participants across novice, intermediate, and advanced lifters |
| **Total Sessions** | **38** trials | Continuous multi-rep experimental sessions (Clean & Jerk) |
| **Total Repetitions** | **186** reps | Standard 5 reps per recording session |
| **Total IMU Samples** | **360,618** samples | Multi-sensor time-series at 100 Hz |
| **Total Duration** | **01:00:06** (MM:SS) | Cumulative recorded kinematics time |
| **Total Storage Size** | **29.35 GB** | Multi-modal bundle across raw, processed, and video files |
| **Nominal IMU Rate** | **100 Hz** | Synchronized 6-DoF accelerometer + gyroscope telemetry |
| **Video Capture Rate** | **30 fps** | Multi-view synchronized Front & Side RGB video captures |

## 2. Participant Cohort & Anthropometrics (Table 1 for Paper)

**Cohort Demographics Summary (Reported N=5):**
- **Age:** 22.2 ± 0.4 years [22 - 23]
- **Body Mass:** 66.6 ± 4.5 kg [60.0 - 72.0]
- **Stature:** 1.70 ± 0.08 m [1.56 - 1.78]
- **Lifter Experience:** 0 to 4 years (Competitive medaled athletes to novices)
- **Gender Distribution:** Predominantly male athletes with female lifter representation (`sub_0005`, 2 trials, 10 reps).
- *Note on Incomplete Questionnaires:* 8 subjects (`sub_0004` to `sub_0010`, `sub_0013`) currently have placeholder demographic parameters awaiting paper finalization.

| entity_id   | name                     |   age |   weight_kg |   height_m |   experience_years | achievement                    | sports_background                     |   total_sessions |   total_reps | metadata_status           |
|:------------|:-------------------------|------:|------------:|-----------:|-------------------:|:-------------------------------|:--------------------------------------|-----------------:|-------------:|:--------------------------|
| sub_0001    | Hivindu Kirulapana       |    22 |          72 |       1.74 |                  1 | -                              | Participation in Other Sports - Rugby |                2 |           10 | ✓ Complete                |
| sub_0002    | Yasith Dissanayake       |    22 |          69 |       1.78 |                  4 | -                              | Participation in Other Sports - Rugby |                5 |           25 | ✓ Complete                |
| sub_0003    | Yewin Dissanayake        |    22 |          65 |       1.7  |                  2 | Intermediate bronze medal 2025 | Participation in Other sports: none   |                5 |           25 | ✓ Complete                |
| sub_0004    | Mahasen                  |     0 |           0 |       0    |                  0 | N\A                            | N\A                                   |                4 |           20 | ⚠️ Incomplete/Placeholder |
| sub_0005    | Achalani                 |     0 |           0 |       0    |                  0 | N\A                            | N\A                                   |                2 |           10 | ⚠️ Incomplete/Placeholder |
| sub_0006    | Pandula                  |     0 |           0 |       0    |                  0 | NA                             | NA                                    |                3 |           15 | ⚠️ Incomplete/Placeholder |
| sub_0007    | Enuka                    |     0 |           0 |       0    |                  0 | NA                             | NA                                    |                2 |           10 | ⚠️ Incomplete/Placeholder |
| sub_0008    | Thevindu                 |     0 |           0 |       0    |                  0 | NA                             | NA                                    |                5 |           21 | ⚠️ Incomplete/Placeholder |
| sub_0009    | Induwara                 |     0 |           0 |       0    |                  0 | NA                             | NA                                    |                2 |           10 | ⚠️ Incomplete/Placeholder |
| sub_0010    | Duleetha                 |     0 |           0 |       0    |                  0 | NA                             | NA                                    |                2 |           10 | ⚠️ Incomplete/Placeholder |
| sub_0011    | Nirmal Samarawickrama    |    22 |          67 |       1.73 |                  0 | -                              | Hockey,wushu                          |                2 |           10 | ✓ Complete                |
| sub_0012    | J. M. Sudeshe Jayasinghe |    23 |          60 |       1.56 |                  0 | -                              | Karate                                |                2 |           10 | ✓ Complete                |
| sub_0013    | Dulan                    |     0 |           0 |       0    |                  0 | N/A                            | N/A                                   |                2 |           10 | ⚠️ Incomplete/Placeholder |

## 3. Storage Hierarchy & Modality Breakdown

### Folder Directory Breakdown

| Folder                     | Size      |   File Count | Percentage   |
|:---------------------------|:----------|-------------:|:-------------|
| related_files/experiments  | 28.90 GB  |          166 | 98.5%        |
| raw_data/experiments       | 352.61 MB |           38 | 1.2%         |
| processed_data/experiments | 73.88 MB  |           78 | 0.2%         |
| raw_data/device            | 35.21 MB  |            1 | 0.1%         |
| summary_reports            | 51.52 KB  |            6 | 0.0%         |
| index                      | 51.16 KB  |            3 | 0.0%         |
| .                          | 6.15 KB   |            1 | 0.0%         |

### File Type Breakdown

| File Type   | Size      |   File Count | Percentage   |
|:------------|:----------|-------------:|:-------------|
| .mp4        | 27.84 GB  |          124 | 94.9%        |
| .mov        | 761.33 MB |            4 | 2.5%         |
| .csv        | 387.85 MB |           42 | 1.3%         |
| .zip        | 320.26 MB |           39 | 1.1%         |
| .parquet    | 73.88 MB  |           78 | 0.2%         |
| .json       | 52.40 KB  |            4 | 0.0%         |
| .md         | 23.95 KB  |            1 | 0.0%         |
| .txt        | 6.15 KB   |            1 | 0.0%         |

## 4. Hardware Sensor Setup & Multi-IMU Node Completeness

The experimental body-area sensor network instruments the anatomical joints as follows:

| Device ID | Code | Anatomical Placement | Primary Role | Expected IMUs | Evaluated Axes |
| :---: | :---: | :--- | :---: | :---: | :--- |
| Device 1 | `RKP` | Right Knee Proximal (Thigh) | Core | 2 (IMUs 1, 2) | AX, AY, AZ, GX, GY, GZ |
| Device 2 | `REP` | Right Elbow Proximal (Upper Arm) | Core | 2 (IMUs 1, 2) | AX, AY, AZ, GX, GY, GZ |
| Device 3 | `LKP` | Left Knee Proximal (Thigh) | Core | 2 (IMUs 1, 2) | AX, AY, AZ, GX, GY, GZ |
| Device 4 | `LEP` | Left Elbow Proximal (Upper Arm) | Core | 2 (IMUs 1, 2) | AX, AY, AZ, GX, GY, GZ |
| Device 5 | `PEL` | Lower Back / Pelvis | Axial/Extended | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |
| Device 6 | `RED` | Right Elbow Distal (Forearm/Wrist) | Core | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |
| Device 7 | `LED` | Left Elbow Distal (Forearm/Wrist) | Core | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |
| Device 8 | `LKD` | Left Knee Distal (Shank) | Core | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |
| Device 9 | `SPN` | Spine / Upper Back | Axial/Extended | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |
| Device 10 | `RKD` | Right Knee Distal (Shank) | Core | 3 (IMUs 1, 2, 3) | AX, AY, AZ, GX, GY, GZ |


### Sensor & IMU Availability Rate Across All Sessions

| Device ID   | Anatomical Code   | Placement                          | Role           | Expected Nodes   | Per-IMU Availability                     | Full Device Health   |
|:------------|:------------------|:-----------------------------------|:---------------|:-----------------|:-----------------------------------------|:---------------------|
| Device 1    | RKP               | Right Knee Proximal (Thigh)        | Core           | 2 IMUs           | IMU1: 100.0%; IMU2: 100.0%               | 100.0%               |
| Device 2    | REP               | Right Elbow Proximal (Upper Arm)   | Core           | 2 IMUs           | IMU1: 100.0%; IMU2: 100.0%               | 100.0%               |
| Device 3    | LKP               | Left Knee Proximal (Thigh)         | Core           | 2 IMUs           | IMU1: 100.0%; IMU2: 92.1%                | 100.0%               |
| Device 4    | LEP               | Left Elbow Proximal (Upper Arm)    | Core           | 2 IMUs           | IMU1: 100.0%; IMU2: 100.0%               | 100.0%               |
| Device 5    | PEL               | Lower Back / Pelvis                | Axial/Extended | 3 IMUs           | IMU1: 100.0%; IMU2: 100.0%; IMU3: 100.0% | 100.0%               |
| Device 6    | RED               | Right Elbow Distal (Forearm/Wrist) | Core           | 3 IMUs           | IMU1: 100.0%; IMU2: 100.0%; IMU3: 100.0% | 100.0%               |
| Device 7    | LED               | Left Elbow Distal (Forearm/Wrist)  | Core           | 3 IMUs           | IMU1: 100.0%; IMU2: 100.0%; IMU3: 97.4%  | 100.0%               |
| Device 8    | LKD               | Left Knee Distal (Shank)           | Core           | 3 IMUs           | IMU1: 100.0%; IMU2: 100.0%; IMU3: 100.0% | 100.0%               |
| Device 9    | SPN               | Spine / Upper Back                 | Axial/Extended | 3 IMUs           | IMU1: 100.0%; IMU2: 100.0%; IMU3: 100.0% | 100.0%               |
| Device 10   | RKD               | Right Knee Distal (Shank)          | Core           | 3 IMUs           | IMU1: 94.7%; IMU2: 94.7%; IMU3: 94.7%    | 94.7%                |

## 5. Multi-View Video & Attachment Completeness

- **Dual-Camera Synchronized Sessions (Front + Side):** 23 / 38 (60.5%)
- **Single-Camera Sessions (Side Only):** 14 / 38 (Trials `experiments_0011` through `experiments_0023`)
- **Single-Camera Sessions (Front Only):** 1 / 38 (Trial `experiments_0007`)
- **Raw Device Archives (.zip):** 38 / 38 (100% complete across all sessions)

## 6. Technical Validation & Known Experimental Anomalies

For complete scientific transparency, the following experimental deviations are cataloged:

1. **Sensor Detachments:**
   - `experiments_0001` (sub_0001): Left ankle sensor detached during the last 3 reps.
   - `experiments_0002` (sub_0001): Left ankle sensor detached during rep 3.

2. **Hardware Exclusions (Device 6 Missing):**
   - `experiments_0003`, `experiments_0004`, `experiments_0005`: Device 6 (Right Elbow Distal / Forearm & Wrist) experienced telemetry disruption and was excluded.

3. **Missing Front Perspective Videos:**
   - Experiments `0011` to `0023` were collected with a single high-speed lateral (Side) camera only.

## 7. Master Experiment Audit Table

Below is the exhaustive, session-by-session inventory.

| entry_id         | entity_id   | subject_name             | date       |   weight_kg |   reps |   row_count | duration_fmt   |   devices_detected_count |   total_imus_detected | video_status            | missing_core_devices   | missing_imus                                                  | diagnostic_flags                                                                                                               |
|:-----------------|:------------|:-------------------------|:-----------|------------:|-------:|------------:|:---------------|-------------------------:|----------------------:|:------------------------|:-----------------------|:--------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------|
| experiments_0001 | sub_0001    | Hivindu Kirulapana       | 2026-05-08 |          20 |      5 |        7713 | 01:17          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Sensor Detachment Noted                                                                                                        |
| experiments_0002 | sub_0001    | Hivindu Kirulapana       | 2026-05-08 |          20 |      5 |        7282 | 01:13          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Sensor Detachment Noted                                                                                                        |
| experiments_0003 | sub_0002    | Yasith Dissanayake       | 2026-05-13 |          20 |      5 |       10263 | 01:43          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Flatline Channels (18 cols) | Note Alert: 5 reps, Device 6 (Right Arm and wrist) are missing                                   |
| experiments_0004 | sub_0002    | Yasith Dissanayake       | 2026-05-13 |          20 |      5 |       10263 | 01:43          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Flatline Channels (18 cols) | Note Alert: 5 reps, Device 6 (Right Arm and wrist) are missing                                   |
| experiments_0005 | sub_0003    | Yewin Dissanayake        | 2026-05-13 |          20 |      5 |       10263 | 01:43          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Flatline Channels (18 cols) | Note Alert: 5 reps, Device 6 (Right Arm and wrist) are missing                                   |
| experiments_0006 | sub_0003    | Yewin Dissanayake        | 2026-05-13 |          20 |      5 |       10263 | 01:43          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Flatline Channels (18 cols)                                                                                                    |
| experiments_0007 | sub_0004    | Mahasen                  | 2026-06-10 |          20 |      5 |       11108 | 01:51          |                       10 |                    26 | Front View Only         | None                   | None                                                          | Missing Side Video                                                                                                             |
| experiments_0008 | sub_0004    | Mahasen                  | 2026-06-10 |          20 |      5 |        7113 | 01:11          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0009 | sub_0004    | Mahasen                  | 2026-06-10 |          20 |      5 |        5027 | 00:50          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0010 | sub_0004    | Mahasen                  | 2026-06-10 |          20 |      5 |        5182 | 00:52          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0011 | sub_0005    | Achalani                 | 2025-10-01 |          20 |      5 |        8402 | 01:24          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0012 | sub_0005    | Achalani                 | 2025-10-01 |          20 |      5 |        8313 | 01:23          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0013 | sub_0006    | Pandula                  | 2025-10-01 |          20 |      5 |       11002 | 01:50          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0014 | sub_0006    | Pandula                  | 2025-10-01 |          20 |      5 |        9778 | 01:38          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0015 | sub_0006    | Pandula                  | 2025-10-01 |          20 |      5 |        1338 | 00:13          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0016 | sub_0007    | Enuka                    | 2025-09-16 |          20 |      5 |       11186 | 01:52          |                        9 |                    23 | Side View Only          | [10]                   | Dev10 (Core) entirely missing (3 IMUs)                        | Missing Core Devices: [10] | Missing IMUs: Dev10 (Core) entirely missing (3 IMUs) | Missing Front Video                        |
| experiments_0017 | sub_0007    | Enuka                    | 2025-09-16 |          20 |      5 |       11885 | 01:59          |                        9 |                    22 | Side View Only          | [10]                   | Dev7 missing IMUs [3]; Dev10 (Core) entirely missing (3 IMUs) | Missing Core Devices: [10] | Missing IMUs: Dev7 missing IMUs [3], Dev10 (Core) entirely missing (3 IMUs) | Missing Front Video |
| experiments_0018 | sub_0008    | Thevindu                 | 2025-09-17 |          20 |      5 |       13993 | 02:20          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0019 | sub_0008    | Thevindu                 | 2025-09-17 |          20 |      5 |       17951 | 03:00          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0020 | sub_0009    | Induwara                 | 2025-10-30 |          20 |      5 |       10817 | 01:48          |                       10 |                    26 | Side View Only          | None                   | None                                                          | NaN Rate: 0.34% | Missing Front Video                                                                                          |
| experiments_0021 | sub_0009    | Induwara                 | 2025-10-30 |          20 |      5 |       12051 | 02:01          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0022 | sub_0010    | Duleetha                 | 2025-10-30 |          20 |      5 |       11437 | 01:54          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0023 | sub_0010    | Duleetha                 | 2025-10-30 |          20 |      5 |       11250 | 01:52          |                       10 |                    26 | Side View Only          | None                   | None                                                          | Missing Front Video                                                                                                            |
| experiments_0024 | sub_0002    | Yasith Dissanayake       | 2026-06-03 |          20 |      5 |        5580 | 00:56          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0025 | sub_0002    | Yasith Dissanayake       | 2026-06-03 |          20 |      5 |        7768 | 01:18          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0026 | sub_0002    | Yasith Dissanayake       | 2026-06-03 |          20 |      5 |        7741 | 01:17          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0027 | sub_0003    | Yewin Dissanayake        | 2026-06-03 |          20 |      5 |       10111 | 01:41          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0028 | sub_0003    | Yewin Dissanayake        | 2026-06-03 |          20 |      5 |        7365 | 01:14          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0029 | sub_0003    | Yewin Dissanayake        | 2026-06-03 |          20 |      5 |       10355 | 01:44          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0030 | sub_0011    | Nirmal Samarawickrama    | 2026-07-17 |          20 |      5 |       12209 | 02:02          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0031 | sub_0011    | Nirmal Samarawickrama    | 2026-07-17 |          20 |      5 |        9546 | 01:35          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0032 | sub_0012    | J. M. Sudeshe Jayasinghe | 2026-07-24 |          20 |      5 |        8390 | 01:24          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0033 | sub_0012    | J. M. Sudeshe Jayasinghe | 2026-07-24 |          20 |      5 |        9006 | 01:30          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Flatline Channels (6 cols)                                                                                                     |
| experiments_0034 | sub_0013    | Dulan                    | 2026-07-24 |          20 |      5 |        9206 | 01:32          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0035 | sub_0013    | Dulan                    | 2026-07-24 |          20 |      5 |        9415 | 01:34          |                       10 |                    26 | Complete (Front + Side) | None                   | None                                                          | Clean / All Checks Passed                                                                                                      |
| experiments_0036 | sub_0008    | Thevindu                 | 2026-08-25 |          20 |      3 |       11796 | 01:58          |                       10 |                    25 | Complete (Front + Side) | None                   | Dev3 missing IMUs [2]                                         | Missing IMUs: Dev3 missing IMUs [2]                                                                                            |
| experiments_0037 | sub_0008    | Thevindu                 | 2026-08-25 |          20 |      5 |       10357 | 01:44          |                       10 |                    25 | Complete (Front + Side) | None                   | Dev3 missing IMUs [2]                                         | Missing IMUs: Dev3 missing IMUs [2]                                                                                            |
| experiments_0038 | sub_0008    | Thevindu                 | 2026-08-25 |          30 |      3 |        7893 | 01:19          |                       10 |                    25 | Side View Only          | None                   | Dev3 missing IMUs [2]                                         | Missing IMUs: Dev3 missing IMUs [2] | Missing Front Video                                                                      |



In [12]:
# @title 📥 Download Individual Deliverable { display-mode: "form" }
# @markdown Select any artifact to download locally.

selected_file = "dataset_paper_bundle.zip" # @param ["dataset_paper_bundle.zip", "dataset_summary_for_paper.md", "experiments_detailed_audit.csv", "subjects_summary.csv", "sensor_completeness_matrix.csv", "dataset_stats.json"]

target_filepath = os.path.join(output_directory, selected_file)

if os.path.exists(target_filepath):
    print(f"File ready: {target_filepath} ({format_bytes(os.path.getsize(target_filepath))})")
    if IN_COLAB:
        files.download(target_filepath)
        print("✅ Download started.")
    else:
        print("💡 Running outside Colab: File is located at", os.path.abspath(target_filepath))
else:
    print(f"❌ File not found: {target_filepath}. Please run the audit pipeline first!")

File ready: ./dataset_paper_summary/dataset_paper_bundle.zip (8.98 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started.
